<a href="https://colab.research.google.com/github/Christianib003/rekomai/blob/clint/notebooks/image_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Data Collection and Processing

In [ ]:
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from deepface import DeepFace
import imgaug.augmenters as iaa

In [ ]:
# 2. Display Original Images
def load_and_display_images(base_path='images/'):
    members = os.listdir(base_path)
    for member in members:
        folder = os.path.join(base_path, member)
        images = [os.path.join(folder, img) for img in os.listdir(folder)]

        plt.figure(figsize=(12, 4))
        for idx, img_path in enumerate(images):
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.subplot(1, len(images), idx+1)
            plt.imshow(img)
            plt.title(os.path.basename(img_path))
            plt.axis('off')
        plt.suptitle(f"{member}'s Expressions")
        plt.show()

load_and_display_images()

In [ ]:
# 3. Define Augmentation Pipeline
augment = iaa.Sequential([
    iaa.Fliplr(1.0),                  # horizontal flip
    iaa.Affine(rotate=(-20, 20)),    # rotate
    iaa.Grayscale(1.0),              # grayscale
])

def apply_augmentations(image):
    return augment(images=[image])[0]

In [ ]:
# 4. Extract Face Embeddings using DeepFace
def extract_embedding(img_path):
    try:
        embedding = DeepFace.represent(img_path=img_path, model_name='Facenet')[0]['embedding']
        return embedding
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return [None] * 128  # fallback placeholder

In [ ]:
# 5. Process and Save Features
data = []

def process_member_images(base_path='images/'):
    for member in os.listdir(base_path):
        folder = os.path.join(base_path, member)
        for img_file in os.listdir(folder):
            img_path = os.path.join(folder, img_file)
            embedding = extract_embedding(img_path)
            data.append({
                'user_id': member,
                'expression': img_file.split('.')[0],
                'image_path': img_path,
                **{f'feat_{i}': val for i, val in enumerate(embedding)}
            })

process_member_images()

In [ ]:
# Convert to DataFrame and Save
image_features_df = pd.DataFrame(data)

# Ensure directory exists
os.makedirs("data/customer-info", exist_ok=True)
image_features_df.to_csv("data/customer-info/image_features.csv", index=False)

# 6. Optional: Preview Feature Dataset
image_features_df.head()